In [ ]:

%pip install -q \
    "langgraph" \
    "crewai" \
    "mlflow" \
    "ragas==0.3.9" \
    "langchain==0.3.27" \
    "langchain-community==0.3.27" \
    "langchain-chroma==0.2.6" \
    "langchain-text-splitters==0.3.9" \
    "transformers==4.56.2" \
    "sentence-transformers" \
    "accelerate" \
    "datasets" \
    "pandas" \
    "scikit-learn" \
    "Pillow==11.3.0" \
    "rapidfuzz"


In [ ]:

import os
import json
import time
import random
import pandas as pd
import torch

import langgraph
import crewai
import mlflow
import ragas
import langchain

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("Python environment ready")
print("PyTorch:", torch.__version__)
print("Ragas:", ragas.__version__)
print("LangChain:", langchain.__version__)
print("CrewAI:", crewai.__version__)
print("CUDA available:", torch.cuda.is_available())


Python environment ready
PyTorch: 2.11.0+cpu
Ragas: 0.3.9
LangChain: 0.3.27
CrewAI: 1.15.28
CUDA available: False


In [ ]:

knowledge_docs = [
    "Machine learning enables computers to learn patterns from data.",
    "Supervised learning trains models using labeled examples.",
    "Unsupervised learning discovers patterns in unlabeled data.",
    "Classification predicts categories, while regression predicts numerical values.",
    "Retrieval-Augmented Generation, or RAG, retrieves relevant information before generating an answer.",
    "A vector database stores vector embeddings and supports similarity-based retrieval.",
    "Embeddings represent text as numerical vectors that capture semantic information.",
    "Chunking splits long documents into smaller pieces for efficient retrieval.",
    "Precision measures the proportion of retrieved results that are relevant.",
    "Recall measures the proportion of relevant results that were retrieved.",
    "Faithfulness measures whether an answer is supported by the retrieved context.",
    "Context precision measures how highly relevant retrieved context is ranked.",
    "Context recall measures whether the retrieved context contains the information needed to answer.",
    "MLflow tracks machine learning experiments, parameters, metrics, and artifacts.",
    "LangGraph builds stateful workflows using nodes and directed edges.",
    "CrewAI coordinates agents and tasks to complete multi-step work.",
    "Ragas evaluates RAG systems using metrics such as faithfulness and answer relevancy."
]

documents = [
    Document(
        page_content=text,
        metadata={"source": f"knowledge_doc_{i + 1}"}
    )
    for i, text in enumerate(knowledge_docs)
]

print("Documents created:", len(documents))
print(documents[0].page_content)


Documents created: 17
Machine learning enables computers to learn patterns from data.


In [ ]:

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Original documents:", len(documents))
print("Total chunks:", len(chunks))
print("\nSample chunk:")
print(chunks[0].page_content)


Original documents: 17
Total chunks: 17

Sample chunk:
Machine learning enables computers to learn patterns from data.


In [ ]:

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={
        "device": "cuda" if torch.cuda.is_available() else "cpu"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

test_embedding = embedding_model.embed_query(
    "What is retrieval-augmented generation?"
)

print("Embedding model loaded successfully.")
print("Embedding dimensions:", len(test_embedding))


/tmp/ipykernel_29955/1047668997.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Embedding model loaded successfully.
Embedding dimensions: 384


In [ ]:

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="w11d5_rag_pipeline"
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("Chroma vector database created.")
print("Chunks indexed:", len(chunks))
print("Retrieved documents per query: 3")


Chroma vector database created.
Chunks indexed: 17
Retrieved documents per query: 3


In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

model.eval()

print("Generator model loaded:", MODEL_NAME)
print("Device:", device)


`torch_dtype` is deprecated! Use `dtype` instead!


Generator model loaded: Qwen/Qwen2.5-0.5B-Instruct
Device: cpu


In [ ]:

def generate_local_answer(prompt, max_new_tokens=120):
    messages = [
        {
            "role": "system",
            "content": (
                "Answer accurately and concisely using the provided context. "
                "Do not invent information."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = output_ids[0][inputs.shape[-1]:]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    return answer


In [ ]:

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class RAGState(TypedDict):
    question: str
    contexts: list[str]
    answer: str


def retrieve_context(state: RAGState):
    docs = retriever.invoke(state["question"])

    return {
        "contexts": [doc.page_content for doc in docs]
    }


def generate_answer(state: RAGState):
    context_text = "\n\n".join(state["contexts"])

    prompt = f"""Answer the question using only the context below.
If the answer is not in the context, say you do not know.

Context:
{context_text}

Question:
{state["question"]}

Answer:"""

    answer = generate_local_answer(prompt)

    return {"answer": answer}


workflow = StateGraph(RAGState)

workflow.add_node("retrieve", retrieve_context)
workflow.add_node("generate", generate_answer)

workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

rag_graph = workflow.compile()

print("LangGraph RAG workflow compiled successfully.")


LangGraph RAG workflow compiled successfully.


/usr/local/lib/python3.13/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [ ]:

test_question = "What is a RAG system?"

test_output = rag_graph.invoke({
    "question": test_question,
    "contexts": [],
    "answer": ""
})

print("Question:", test_question)
print("\nRetrieved contexts:")

for i, context in enumerate(test_output["contexts"], start=1):
    print(f"{i}. {context}")

print("\nGenerated answer:")
print(test_output["answer"])


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Question: What is a RAG system?

Retrieved contexts:
1. Ragas evaluates RAG systems using metrics such as faithfulness and answer relevancy.
2. Retrieval-Augmented Generation, or RAG, retrieves relevant information before generating an answer.
3. Chunking splits long documents into smaller pieces for efficient retrieval.

Generated answer:
A RAG system is a type of machine learning model that evaluates RAG (Relevance-Aggregation) systems based on metrics like faithfulness and answer relevance.


In [ ]:

qa_data = [
    {
        "question": "What is machine learning?",
        "ground_truth": "Machine learning enables computers to learn patterns from data."
    },
    {
        "question": "What is supervised learning?",
        "ground_truth": "Supervised learning trains models using labeled examples."
    },
    {
        "question": "What is unsupervised learning?",
        "ground_truth": "Unsupervised learning discovers patterns in unlabeled data."
    },
    {
        "question": "What is a RAG system?",
        "ground_truth": "RAG retrieves relevant information before generating an answer."
    },
    {
        "question": "What is a vector database?",
        "ground_truth": "A vector database stores embeddings and supports similarity-based retrieval."
    },
    {
        "question": "What are embeddings?",
        "ground_truth": "Embeddings represent text as numerical vectors that capture semantic information."
    },
    {
        "question": "Why is chunking used in RAG?",
        "ground_truth": "Chunking splits long documents into smaller pieces for efficient retrieval."
    },
    {
        "question": "What does precision measure?",
        "ground_truth": "Precision measures the proportion of retrieved results that are relevant."
    },
    {
        "question": "What does recall measure?",
        "ground_truth": "Recall measures the proportion of relevant results that were retrieved."
    },
    {
        "question": "What is MLflow used for?",
        "ground_truth": "MLflow tracks machine learning experiments, parameters, metrics, and artifacts."
    }
]

print("Evaluation questions:", len(qa_data))


Evaluation questions: 10


In [ ]:

results = []

for i, item in enumerate(qa_data, start=1):
    output = rag_graph.invoke({
        "question": item["question"],
        "contexts": [],
        "answer": ""
    })

    results.append({
        "question": item["question"],
        "answer": output["answer"],
        "contexts": output["contexts"],
        "ground_truth": item["ground_truth"]
    })

    print(f"{i}/10 completed")

rag_df = pd.DataFrame(results)
rag_df.to_csv("w11d5_rag_outputs.csv", index=False)

display(rag_df[["question", "answer", "ground_truth"]].head())
print("Saved w11d5_rag_outputs.csv")


1/10 completed
2/10 completed
3/10 completed
4/10 completed
5/10 completed
6/10 completed
7/10 completed
8/10 completed
9/10 completed
10/10 completed


,question,answer,ground_truth
0,What is machine learning?,Machine learning is a subset of artificial int...,Machine learning enables computers to learn pa...
1,What is supervised learning?,Supervised learning trains models using labele...,Supervised learning trains models using labele...
2,What is unsupervised learning?,Unsupervised learning discovers patterns in un...,Unsupervised learning discovers patterns in un...
3,What is a RAG system?,A RAG system is a type of machine learning mod...,RAG retrieves relevant information before gene...
4,What is a vector database?,A vector database stores vector embeddings and...,A vector database stores embeddings and suppor...


Saved w11d5_rag_outputs.csv


In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM
from langchain_core.language_models.llms import LLM
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

EVALUATOR_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

evaluator_tokenizer = AutoTokenizer.from_pretrained(
    EVALUATOR_MODEL_NAME
)

evaluator_model = AutoModelForCausalLM.from_pretrained(
    EVALUATOR_MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
).to(device)

evaluator_model.eval()

print("Evaluator model loaded:", EVALUATOR_MODEL_NAME)


Evaluator model loaded: Qwen/Qwen2.5-1.5B-Instruct


In [ ]:

class LocalEvaluatorLLM(LLM):
    @property
    def _llm_type(self):
        return "local-qwen-evaluator"

    @property
    def _identifying_params(self):
        return {"model": EVALUATOR_MODEL_NAME}

    def _call(self, prompt, stop=None, run_manager=None, **kwargs):
        messages = [
            {
                "role": "system",
                "content": (
                    "You are a careful RAG evaluation assistant. "
                    "Follow the requested output format exactly."
                )
            },
            {"role": "user", "content": prompt}
        ]

        inputs = evaluator_tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            output_ids = evaluator_model.generate(
                inputs,
                max_new_tokens=512,
                do_sample=False,
                pad_token_id=evaluator_tokenizer.eos_token_id
            )

        new_tokens = output_ids[0][inputs.shape[-1]:]
        answer = evaluator_tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip()

        if stop:
            for stop_text in stop:
                if stop_text in answer:
                    answer = answer.split(stop_text)[0]

        return answer


local_evaluator = LocalEvaluatorLLM()
evaluator_llm = LangchainLLMWrapper(local_evaluator)
evaluator_embeddings = LangchainEmbeddingsWrapper(embedding_model)

print("Ragas evaluator wrappers configured.")


Ragas evaluator wrappers configured.


/tmp/ipykernel_29955/2934509209.py:52: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(local_evaluator)
/tmp/ipykernel_29955/2934509209.py:53: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(embedding_model)


In [ ]:

evaluator_test = local_evaluator.invoke(
    "Reply with exactly this text: EVALUATOR_READY",
    config={"stop": []}
)

print("Evaluator test output:")
print(evaluator_test)


Evaluator test output:
EVALUATOR_READY


In [ ]:

from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

metrics = [
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
]

for metric in metrics:
    if hasattr(metric, "llm"):
        metric.llm = evaluator_llm
    if hasattr(metric, "embeddings"):
        metric.embeddings = evaluator_embeddings

print("Configured metrics:")
for metric in metrics:
    print("-", metric.name)


Configured metrics:
- faithfulness
- answer_relevancy
- context_precision
- context_recall


In [ ]:

from datasets import Dataset
from ragas import evaluate
from ragas.run_config import RunConfig

ragas_dataset = Dataset.from_dict({
    "question": rag_df["question"].tolist(),
    "answer": rag_df["answer"].tolist(),
    "contexts": rag_df["contexts"].tolist(),
    "ground_truth": rag_df["ground_truth"].tolist()
})

print("Dataset rows:", len(ragas_dataset))
print("Dataset columns:", ragas_dataset.column_names)
print("Missing answers:", int(rag_df["answer"].isna().sum()))
print("Empty contexts:", int(
    rag_df["contexts"].apply(
        lambda x: not isinstance(x, list) or len(x) == 0
    ).sum()
))


Dataset rows: 10
Dataset columns: ['question', 'answer', 'contexts', 'ground_truth']
Missing answers: 0
Empty contexts: 0


In [ ]:

from datasets import Dataset
from ragas import evaluate
from ragas.run_config import RunConfig

# Recreate the Ragas dataset from the existing results
ragas_dataset = Dataset.from_dict({
    "question": rag_df["question"].astype(str).tolist(),
    "answer": rag_df["answer"].astype(str).tolist(),
    "contexts": [
        [str(context) for context in row]
        if isinstance(row, (list, tuple))
        else [str(row)]
        for row in rag_df["contexts"].tolist()
    ],
    "ground_truth": rag_df["ground_truth"].astype(str).tolist()
})

print("Ragas dataset created successfully.")
print("Rows:", len(ragas_dataset))
print("Columns:", ragas_dataset.column_names)


Ragas dataset created successfully.
Rows: 10
Columns: ['question', 'answer', 'contexts', 'ground_truth']


In [ ]:

# Allow each Ragas metric up to 30 minutes
for metric in metrics:
    metric.timeout = 1800

print("Metric timeouts updated:")
for metric in metrics:
    print(metric.name, ":", metric.timeout, "seconds")


Metric timeouts updated:
faithfulness : 1800 seconds
answer_relevancy : 1800 seconds
context_precision : 1800 seconds
context_recall : 1800 seconds


In [ ]:

test_dataset = ragas_dataset.select([0])

test_result = evaluate(
    dataset=test_dataset,
    metrics=[faithfulness],
    run_config=RunConfig(
        timeout=1800,
        max_retries=1,
        max_workers=1
    ),
    raise_exceptions=True
)

print("Single-example faithfulness test completed.")
display(test_result.to_pandas())


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

Single-example faithfulness test completed.


,user_input,retrieved_contexts,response,reference,faithfulness
0,What is machine learning?,[Machine learning enables computers to learn p...,Machine learning is a subset of artificial int...,Machine learning enables computers to learn pa...,1.0


In [27]:

ragas_result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics,
    run_config=RunConfig(
        timeout=1800,
        max_retries=1,
        max_workers=1
    ),
    raise_exceptions=True
)

evaluation_df = ragas_result.to_pandas()

print("Full Ragas evaluation completed.")
display(evaluation_df)


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

Full Ragas evaluation completed.


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is machine learning?,[Machine learning enables computers to learn p...,Machine learning is a subset of artificial int...,Machine learning enables computers to learn pa...,1.0,0.891225,0.833333,1.0
1,What is supervised learning?,[Supervised learning trains models using label...,Supervised learning trains models using labele...,Supervised learning trains models using labele...,1.0,0.887042,1.000000,1.0
2,What is unsupervised learning?,[Unsupervised learning discovers patterns in u...,Unsupervised learning discovers patterns in un...,Unsupervised learning discovers patterns in un...,1.0,0.693005,1.000000,1.0
3,What is a RAG system?,[Ragas evaluates RAG systems using metrics suc...,A RAG system is a type of machine learning mod...,RAG retrieves relevant information before gene...,0.5,1.000000,1.000000,1.0
4,What is a vector database?,[A vector database stores vector embeddings an...,A vector database stores vector embeddings and...,A vector database stores embeddings and suppor...,1.0,0.900945,1.000000,1.0
5,What are embeddings?,[Embeddings represent text as numerical vector...,embeddings represent text as numerical vectors...,Embeddings represent text as numerical vectors...,1.0,0.939333,1.000000,1.0
6,Why is chunking used in RAG?,[Chunking splits long documents into smaller p...,Chunking is used in RAG to split long document...,Chunking splits long documents into smaller pi...,1.0,0.937435,1.000000,1.0
7,What does precision measure?,[Precision measures the proportion of retrieve...,precision measures the proportion of retrieved...,Precision measures the proportion of retrieved...,1.0,0.771876,1.000000,1.0
8,What does recall measure?,[Recall measures the proportion of relevant re...,recall measures the proportion of relevant res...,Recall measures the proportion of relevant res...,1.0,0.834025,1.000000,1.0
9,What is MLflow used for?,"[MLflow tracks machine learning experiments, p...",Machine learning enables computers to learn pa...,"MLflow tracks machine learning experiments, pa...",1.0,0.250904,0.833333,1.0


In [28]:

evaluation_df.to_csv(
    "w11d5_ragas_evaluation.csv",
    index=False
)

score_columns = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall"
]

available_scores = [
    col for col in score_columns
    if col in evaluation_df.columns
]

print("Saved: w11d5_ragas_evaluation.csv")
print("\nAverage metric scores:")

display(
    evaluation_df[available_scores]
    .apply(pd.to_numeric, errors="coerce")
    .mean()
    .to_frame("mean_score")
)

print("\nEvaluation rows:", len(evaluation_df))


Saved: w11d5_ragas_evaluation.csv

Average metric scores:


,mean_score
faithfulness,0.950000
answer_relevancy,0.810579
context_precision,0.966667
context_recall,1.000000



Evaluation rows: 10


In [29]:

import pandas as pd
import os

assert "rag_df" in globals(), "RAG output dataset is missing."
assert "evaluation_df" in globals(), "Ragas evaluation did not finish."

assert len(rag_df) == 10, f"Expected 10 answers, found {len(rag_df)}."
assert len(evaluation_df) == 10, (
    f"Expected 10 evaluation rows, found {len(evaluation_df)}."
)

expected_metrics = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall"
]

available_metrics = [
    name for name in expected_metrics
    if name in evaluation_df.columns
]

print("RAG answers:", len(rag_df))
print("Evaluation rows:", len(evaluation_df))
print("Available metrics:", available_metrics)
print("Evaluation columns:", evaluation_df.columns.tolist())

assert len(available_metrics) == 4, (
    "Not all four metrics are present. Check the evaluation output."
)

display(
    evaluation_df[available_metrics]
    .apply(pd.to_numeric, errors="coerce")
    .mean()
    .to_frame("mean_score")
)


RAG answers: 10
Evaluation rows: 10
Available metrics: ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']
Evaluation columns: ['user_input', 'retrieved_contexts', 'response', 'reference', 'faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']


,mean_score
faithfulness,0.950000
answer_relevancy,0.810579
context_precision,0.966667
context_recall,1.000000


In [30]:

from crewai import BaseLLM, Agent, Task, Crew, Process

class CrewAIQwenLLM(BaseLLM):
    def __init__(self):
        super().__init__(
            model=EVALUATOR_MODEL_NAME,
            temperature=0.0
        )

    def call(
        self,
        messages,
        tools=None,
        callbacks=None,
        available_functions=None,
        from_task=None,
        from_agent=None,
        response_model=None
    ):
        if isinstance(messages, str):
            prompt = messages
        else:
            parts = []
            for message in messages:
                if isinstance(message, dict):
                    role = message.get("role", "user")
                    content = message.get("content", "")
                else:
                    role = getattr(message, "role", "user")
                    content = getattr(message, "content", str(message))

                parts.append(f"{role}: {content}")

            prompt = "\n".join(parts)

        return local_evaluator.invoke(prompt)

    def supports_function_calling(self):
        return False


crewai_llm = CrewAIQwenLLM()

reviewer_agent = Agent(
    role="RAG Quality Reviewer",
    goal="Identify answer quality and grounding issues",
    backstory=(
        "You review retrieval-augmented generation systems for "
        "factual consistency, relevance, and unsupported claims."
    ),
    llm=crewai_llm,
    allow_delegation=False,
    max_iter=1,
    verbose=False
)

print("CrewAI review agent created.")


CrewAI review agent created.


In [33]:

import asyncio

async def run_review_crew():
    return await review_crew.kickoff_async()

crew_review = await run_review_crew()
crew_review_text = str(crew_review)

with open("w11d5_crew_review.txt", "w", encoding="utf-8") as f:
    f.write(crew_review_text)

print("CrewAI review completed.")
print(crew_review_text[:4000])


CrewAI review completed.
### Question 1:
**Generated Answer:**  
Machine learning is a subset of artificial intelligence that involves algorithms that enable computers to learn from data without being explicitly programmed. It enables machines to analyze large amounts of information automatically and make predictions or decisions based on patterns found within that data. Machine learning can be applied to various fields such as image recognition, natural language processing, recommendation systems, and more.

**Analysis:**  
- **Relevance:** The generated answer is relevant to the question "What is machine learning?" but it does not provide a comprehensive definition. It mentions subsets (artificial intelligence) and applications, which could be expanded upon.
- **Support:** The answer supports the retrieved context by mentioning that machine learning involves learning from data without explicit programming.
- **Unsupported Claim:** There are no unsupported claims in this answer.

**Im

In [37]:
import mlflow
import numpy as np
import pandas as pd
import os

# FIX: Use SQLite instead of the old file-based tracking backend
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("W11D5_Tracked_RAG_Pipeline")

with mlflow.start_run(run_name="local_rag_evaluation"):

    mlflow.log_param("generator_model", MODEL_NAME)
    mlflow.log_param("evaluator_model", EVALUATOR_MODEL_NAME)
    mlflow.log_param(
        "embedding_model",
        "sentence-transformers/all-MiniLM-L6-v2"
    )
    mlflow.log_param("retrieval_k", 3)
    mlflow.log_param("evaluation_rows", len(evaluation_df))
    mlflow.log_param("rag_framework", "LangGraph")
    mlflow.log_param("review_framework", "CrewAI")
    mlflow.log_param("evaluation_framework", "Ragas")

    for metric_name in available_metrics:
        scores = pd.to_numeric(
            evaluation_df[metric_name],
            errors="coerce"
        ).dropna()

        if not scores.empty:
            mlflow.log_metric(
                f"mean_{metric_name}",
                float(scores.mean())
            )

    mlflow.log_artifact("w11d5_rag_outputs.csv")
    mlflow.log_artifact("w11d5_ragas_evaluation.csv")
    mlflow.log_artifact("w11d5_crew_review.txt")

print("MLflow tracking completed.")
print("MLflow database:", os.path.abspath("./mlflow.db"))
print("MLflow directory:", os.path.abspath("./mlartifacts"))

2026/10/14 17:22:40 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/14 17:22:40 INFO mlflow.store.db.utils: Updating database tables
2026/10/14 17:22:43 INFO mlflow.tracking.fluent: Experiment with name 'W11D5_Tracked_RAG_Pipeline' does not exist. Creating a new experiment.


MLflow tracking completed.
MLflow database: /content/mlflow.db
MLflow directory: /content/mlartifacts


In [38]:
import mlflow
import numpy as np
import pandas as pd
import os

# FIX: Use SQLite instead of the old file-based tracking backend
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("W11D5_Tracked_RAG_Pipeline")

with mlflow.start_run(run_name="local_rag_evaluation"):

    mlflow.log_param("generator_model", MODEL_NAME)
    mlflow.log_param("evaluator_model", EVALUATOR_MODEL_NAME)
    mlflow.log_param(
        "embedding_model",
        "sentence-transformers/all-MiniLM-L6-v2"
    )
    mlflow.log_param("retrieval_k", 3)
    mlflow.log_param("evaluation_rows", len(evaluation_df))
    mlflow.log_param("rag_framework", "LangGraph")
    mlflow.log_param("review_framework", "CrewAI")
    mlflow.log_param("evaluation_framework", "Ragas")

    for metric_name in available_metrics:
        scores = pd.to_numeric(
            evaluation_df[metric_name],
            errors="coerce"
        ).dropna()

        if not scores.empty:
            mlflow.log_metric(
                f"mean_{metric_name}",
                float(scores.mean())
            )

    mlflow.log_artifact("w11d5_rag_outputs.csv")
    mlflow.log_artifact("w11d5_ragas_evaluation.csv")
    mlflow.log_artifact("w11d5_crew_review.txt")

print("MLflow tracking completed.")
print("MLflow database:", os.path.abspath("./mlflow.db"))
print("MLflow directory:", os.path.abspath("./mlartifacts"))

MLflow tracking completed.
MLflow database: /content/mlflow.db
MLflow directory: /content/mlartifacts


In [39]:
from google.colab import files

files.download("w11d5_rag_outputs.csv")
files.download("w11d5_ragas_evaluation.csv")
files.download("w11d5_crew_review.txt")
files.download("mlflow.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>